In [ ]:

from collections import Counter
from collections import defaultdict

# Base
import math as m

# Base 2.0
import numpy as np
import pandas as pd

# Grafici
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# SciPy
from scipy import stats
from scipy.cluster.hierarchy import (
    dendrogram,
    fcluster,
    linkage,
)
from scipy.spatial.distance import pdist, squareform

# Scikit-learn
from sklearn.cluster import (
    AgglomerativeClustering,
    BisectingKMeans,
    cluster_optics_dbscan,
    DBSCAN,
    KMeans,
    OPTICS,
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    RidgeCV,
    Lasso,
    LassoCV,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
    r2_score,
    RocCurveDisplay,
    silhouette_score,
    silhouette_samples,
)
from sklearn.model_selection import (
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import (
    CategoricalNB,
    GaussianNB,
)
from sklearn.neighbors import (
    KNeighborsClassifier,
    NearestNeighbors,
    KNeighborsRegressor,
)
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
)
from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
    plot_tree,
)
from pyod.models.knn import KNN
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm

# Librerie
import pandas as pd
import numpy as np

# High Dimensionality
from pyod.models.abod import ABOD

# Ensemble
from pyod.models.feature_bagging import FeatureBagging
# carico LOF per FeaBag, chiaramente si può cambiare a seconda del modello semplice scelto
from pyod.models.lof import LOF
from pyod.models.loda import LODA

# Foreste
from sklearn.ensemble import IsolationForest
from isotree import IsolationForest as EIF

from sklearn.manifold import TSNE


# REGRESSIONE
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor

In [ ]:
# dataset
df = pd.read_parquet("df_pulito.parquet")
# df["bmi"] = (df["Physical-Weight"] / (df["Physical-Height"] ** 2)) * 703

# Train Test sii
pciat_cols = [f"PCIAT-PCIAT_{i:02d}" for i in range(1, 21)]
X = df.drop(columns=["sii", "risultati",
            "PCIAT-PCIAT_Total", "id"] + pciat_cols)
y = df["sii"]  # per valutare sii, praticamente classificazione
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=222)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Test Ore
X_ore = df.drop(columns=["risultati", "id",
                "PreInt_EduHx-computerinternet_hoursday"])
y_ore = df['PreInt_EduHx-computerinternet_hoursday']

X_train_ore, X_test_ore, y_train_ore, y_test_ore = train_test_split(
    X_ore, y_ore, test_size=0.3, random_state=222)


# Train Test Attività Fisica
X_af = df.drop(columns=["risultati", "BIA-BIA_Activity_Level_num", "id"])
y_af = df['BIA-BIA_Activity_Level_num']

X_train_af, X_test_af, y_train_af, y_test_af = train_test_split(
    X_af, y_af, test_size=0.3, random_state=222)

In [ ]:
################################
# CLASSIFICAZIONE

In [ ]:
# Logistica
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

bruh = {0.0: 1, 1.0: 1, 2.0: 1, 3.0: 1}

logistica = LogisticRegression(
    random_state=222, max_iter=100, class_weight=bruh)
logistica = LogisticRegression(
    random_state=222,
    max_iter=100,
    class_weight=bruh,
    multi_class='ovr'
)
logistica.fit(X_train_scaled, y_train)

y_pred_log = logistica.predict(X_test_scaled)

print('Accuracy %s' % accuracy_score(y_test, y_pred_log))
print('F1-score %s' % f1_score(y_test, y_pred_log, average=None))
print(classification_report(y_test, y_pred_log))

In [ ]:
# SVM
from sklearn.svm import SVC
from sklearn.svm import LinearSVC

# Linear
svmL = LinearSVC(C=0.001, random_state=222)
svmL.fit(X_train, y_train)

y_pred_svmL = svmL.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_svmL))
print('F1-score %s' % f1_score(y_test, y_pred_svmL, average=None))
print(classification_report(y_test, y_pred_svmL))

# Kernel
svmK = SVC(gamma='auto', C=0.1, kernel='rbf', random_state=222)
svmK.fit(X_train, y_train)

y_pred_svmK = svmK.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_svmK))
print('F1-score %s' % f1_score(y_test, y_pred_svmK, average=None))
print(classification_report(y_test, y_pred_svmK))

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

X = X[y != 0, :2]
y = y[y != 0]

n_sample = len(X)

np.random.seed(0)
order = np.random.permutation(n_sample)
X = X[order]
y = y[order].astype(np.float)

X_train = X[:int(.9 * n_sample)]
y_train = y[:int(.9 * n_sample)]
X_test = X[int(.9 * n_sample):]
y_test = y[int(.9 * n_sample):]

#fit the model
for kernel in ('linear', 'rbf', 'poly'):
    clf = SVC(kernel=kernel, gamma=10)
    clf.fit(X_train, y_train)

    plt.figure()
    plt.clf()
    plt.scatter(X[:, 0], X[:, 1], c=y, zorder=10, cmap=plt.cm.Paired,
                edgecolor='k', s=20)

    # Circle out the test data
    plt.scatter(X_test[:, 0], X_test[:, 1], s=80, facecolors='none',
                zorder=10, edgecolor='k')

    plt.axis('tight')
    x_min = X[:, 0].min()
    x_max = X[:, 0].max()
    y_min = X[:, 1].min()
    y_max = X[:, 1].max()

    XX, YY = np.mgrid[x_min:x_max:200j, y_min:y_max:200j]
    Z = clf.decisionfunction(np.c[XX.ravel(), YY.ravel()])

    # Put the result into a color plot
    Z = Z.reshape(XX.shape)
    plt.pcolormesh(XX, YY, Z > 0, cmap=plt.cm.Paired)
    plt.contour(XX, YY, Z, colors=['k', 'k', 'k'],
                linestyles=['--', '-', '--'], levels=[-.5, 0, .5])

    plt.title(kernel)
plt.show()

In [ ]:
# ENSEMBLE

# RINOMINARE y_pred
from sklearn.ensemble import BaggingClassifier

bagging = BaggingClassifier(estimator=SVC(C=1000),
                            n_estimators=10, random_state=222)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred))
print('F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

bagging = BaggingClassifier(estimator=RandomForestClassifier(n_estimators=100),
                            n_estimators=10, random_state=222)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred))
print('F1-score %s' % f1_score(y_test, y_pred, average=None))
print(classification_report(y_test, y_pred))

In [ ]:
# Gradiente
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
AdaBoost = AdaBoostClassifier(estimator=RandomForestClassifier(
    n_estimators=100), n_estimators=100, random_state=222)
AdaBoost.fit(X_train, y_train)

y_pred_ABoost = AdaBoost.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_ABoost))
print('F1-score %s' % f1_score(y_test, y_pred_ABoost, average=None))
print(classification_report(y_test, y_pred_ABoost))

# Gradient Boost
GraBoost = GradientBoostingClassifier(n_estimators=100,

                                      learning_rate=1.0, max_depth=3, random_state=0)

GraBoost.fit(X_train, y_train)
y_pred_GBoost = GraBoost.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_GBoost))

print('F1-score %s' % f1_score(y_test, y_pred_GBoost, average=None))

print(classification_report(y_test, y_pred_GBoost))

# XGBOOST
XGBoost = XGBClassifier(objective='binary:logistic',
                        max_depth=6,
                        learning_rate=1.0,
                        gamma=0.0,
                        reg_lambda=1,
                        tree_method='exact',  # 'approx'
                        use_label_encoder=False,
                        random_state=42
                        )
XGBoost.fit(X_train, y_train)

y_pred_XGBoost = XGBoost.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_XGBoost))
print('F1-score %s' % f1_score(y_test, y_pred_XGBoost, average=None))
print(classification_report(y_test, y_pred_XGBoost))

# LightGBM

Light = LGBMClassifier(boosting_type='gbdt',  # 'goss', #'dart'
                       max_depth=-1,  # no limit
                       num_leaves=31,
                       n_estimators=100,
                       subsample_for_bin=200000,
                       objective='multiclass',
                       reg_alpha=0.0,  # L1 regularization term on weights
                       reg_lambda=0.0,  # L2 regularization term on weights
                       random_state=42
                       )
Light.fit(X_train, y_train)

y_pred_Light = Light.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_Light))
print('F1-score %s' % f1_score(y_test, y_pred_Light, average=None))
print(classification_report(y_test, y_pred_Light))

# CatBoost
CBoost = CatBoostClassifier()
CBoost.fit(X_train, y_train)
y_pred_CBoost = CBoost.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_CBoost))
print('F1-score %s' % f1_score(y_test, y_pred_CBoost, average=None))
print(classification_report(y_test, y_pred_CBoost))

# Explainable non trova il pacchetto
'''# Explainable
from interpret.glassbox import ExplainableBoostingClassifier
from interpret import show
ExplainableBoost = ExplainableBoostingClassifier()
ExplainableBoost.fit(X_train, y_train)

y_pred_ExpBoost = CBoost.predict(X_test)

print('Accuracy %s' % accuracy_score(y_test, y_pred_ExpBoost))
print('F1-score %s' % f1_score(y_test, y_pred_ExpBoost, average=None))
print(classification_report(y_test, y_pred_ExpBoost))'''